In [11]:
import pandas as pd
import numpy as np
from calculate_immune_signature import ImmuneSignatureCalculator
# or_data = pd.read_csv('../../Data/original_data.csv')
seeds = [0,1,2,3,42]
df_signatures = pd.read_excel('1-s2.0-S0092867418311784-mmc3.xlsx', sheet_name=1, header=1, usecols='A:Q')


signatures ={'B CELL': df_signatures['B CELL'].dropna().tolist(),
             'MACROPHAGE': df_signatures['MACROPHAGE'].dropna().tolist(),
             'CD4': df_signatures['T CD4'].dropna().tolist(),
             'CD8': df_signatures['T CD8'].dropna().tolist(),
             'T CELL': df_signatures['T CELL'].dropna().tolist(),
             'IMMUNE': df_signatures['IMMUNE'].dropna().tolist(),
             'T CD4 EXHAUSTED': df_signatures['T CD4 EXHAUSTED'].dropna().tolist(),
             'T CD4 TREG': df_signatures['T CD4 TREG'].dropna().tolist(),
             'T CD8 CYTOTOXIC': df_signatures['T CD8 CYTOTOXIC'].dropna().tolist(),
             'T CD8 EXHAUSTED': df_signatures['T CD8 EXHAUSTED'].dropna().tolist(),
}
for seed in seeds: 
    datasets = [f'avatarsk5_{seed}', f'avatarsk10_{seed}',f'ctgan_{seed}',
           f'gaussiancopula_{seed}', f'synthpop_{seed}', f'tvae_{seed}']
    for dataset in datasets:
        print(f'---{dataset}---')
        data = pd.read_csv(f'../../Data/{dataset}.csv').set_index('Patient_ID')
        expression_matrix = data.iloc[:,54:]
        expression_matrix = expression_matrix.T
        expression_matrix = (expression_matrix
                             .replace([np.inf, -np.inf], 0)
                             .fillna(0)
                             .clip(lower=0))
        
        # Verify
        assert not np.isinf(expression_matrix.values).any(), "Still has inf!"
        assert not np.isnan(expression_matrix.values).any(), "Still has NaN!"
        assert (expression_matrix.values >= 0).all(), "Has negative values!"
        
        print("✅ Data cleaned successfully!")
        
        # Calculate
        calculator = ImmuneSignatureCalculator(
            expression_matrix=expression_matrix,
            signatures=signatures,
            num_rounds=1000,
            n_bins=50,
            random_seed=42,
            is_logged=False
        )

        
        scores = calculator.calculate_signatures()
        scores.to_csv(f'Jerby_ImmuneCell/{dataset}.csv')

---avatarsk5_0---
✅ Data cleaned successfully!
Converting raw TPM to log2(TPM+1)...
Initialized with 18760 genes, 121 samples

CALCULATING SIGNATURE SCORES

Step 1: Calculating z-scores...
  Mean expression range: [0.030, 15.685]

Step 2: Binning genes by expression...
  Created 50 bins

Step 3: Calculating signature scores...

  [1/10] Processing:  B CELL
    Found 76/91 genes
    Score range: [-0.629, 1.350]

  [2/10] Processing:  MACROPHAGE
    Found 428/446 genes
    Score range: [-0.680, 0.858]

  [3/10] Processing:  CD4
    Found 44/50 genes
    Score range: [-0.675, 1.281]

  [4/10] Processing:  CD8
    Found 46/50 genes
    Score range: [-0.775, 1.678]

  [5/10] Processing:  T CELL
    Found 102/110 genes
    Score range: [-0.956, 1.812]

  [6/10] Processing:  IMMUNE
    Found 98/100 genes
    Score range: [-1.313, 1.913]

  [7/10] Processing:  T CD4 EXHAUSTED
    Found 27/31 genes
    Score range: [-0.461, 0.702]

  [8/10] Processing:  T CD4 TREG
    Found 38/39 genes
    Scor